# Spotify Dataset Analysis — The Smiths

**Trabajo individual basado en el fichero de ejemplo `Spotify_simple_Colab`.**

Este notebook explica el proyecto de forma ordenada y deja visibles los resultados principales: tablas, gráficos musicales, análisis de letras y referencia a la web app.

El objetivo es cumplir los tres apartados del enunciado:

1. Filtrar álbumes para conservar únicamente álbumes de estudio.
2. Incorporar gráficos relacionados con letras de canciones.
3. Crear una web app con los componentes implementados.

## 1. Dataset utilizado

Se utiliza el dataset **Spotify 1.2M+ Songs**, cuyo fichero principal es `tracks_features.csv`.

Este dataset contiene canciones y variables musicales como:

| Variable | Descripción |
|---|---|
| `name` | Nombre de la canción |
| `album` | Álbum |
| `artists` | Artistas asociados |
| `artist_ids` | Identificadores de artistas |
| `energy` | Intensidad o energía musical |
| `valence` | Positividad musical |
| `danceability` | Bailabilidad |
| `acousticness` | Grado acústico |
| `tempo` | Velocidad en BPM |
| `duration_ms` | Duración en milisegundos |

El script `analysis_cursor.py` intenta descargarlo automáticamente si no existe en `data/raw/`.

## 2. Preparación del entorno

Si se ejecuta en local, antes se debe crear y activar el entorno virtual:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

Si se ejecuta en Colab y falta alguna librería, se puede descomentar la línea de instalación de la siguiente celda.

In [ ]:
# En Colab, si faltan librerías, descomenta esta línea:
# !pip install pandas matplotlib requests wordcloud gdown streamlit plotly -q

from pathlib import Path
import os
import pandas as pd
from IPython.display import display, Image

## 3. Ejecución del análisis principal

El análisis completo está implementado en `analysis_cursor.py`. Este script:

- carga el dataset;
- filtra el artista **The Smiths** con coincidencia exacta;
- evita falsos positivos como `The Smithsonian Chamber Players`;
- compara los álbumes encontrados con la discografía oficial;
- conserva únicamente álbumes de estudio;
- genera gráficos musicales por canción;
- consulta letras con `lyrics.ovh`;
- si no encuentra letras de The Smiths, usa un conjunto alternativo de artistas;
- guarda resultados en `data/processed/` y gráficos en `outputs/plots/`.

La siguiente celda ejecuta el script desde el notebook.

In [ ]:
# Localizamos la raíz del proyecto aunque el notebook se abra desde /notebook
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analysis_cursor.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
print("Proyecto:", PROJECT_ROOT)

# Ejecuta el análisis completo
exec(open(PROJECT_ROOT / "analysis_cursor.py", encoding="utf-8").read())

## 4. Filtrado de álbumes de estudio

Según la discografía oficial del enunciado, los álbumes de estudio de **The Smiths** son:

1. *The Smiths*
2. *Meat Is Murder*
3. *The Queen Is Dead*
4. *Strangeways, Here We Come*

Durante el proyecto se comprobó que una búsqueda textual simple con `str.contains("The Smiths")` no era suficiente, porque también detectaba artistas como `The Smithsonian Chamber Players`. Por eso se usa una comprobación exacta del artista dentro de la lista `artists`.

Tras aplicar el filtro exacto y conservar solo álbumes de estudio, el dataset solo mantiene canciones del álbum *Strangeways, Here We Come*.

In [ ]:
album_counts_path = PROJECT_ROOT / "data" / "processed" / "conteo_albumes_estudio_the_smiths.csv"
studio_tracks_path = PROJECT_ROOT / "data" / "processed" / "the_smiths_studio_tracks.csv"

album_counts = pd.read_csv(album_counts_path)
studio_tracks = pd.read_csv(studio_tracks_path)

print("Conteo de canciones por álbum de estudio:")
display(album_counts)

print("Canciones conservadas después del filtrado:")
display(studio_tracks[["name", "short_album_name", "album", "year", "track_number"]])

### Interpretación del filtrado

Aunque The Smiths tiene cuatro álbumes de estudio, el dataset utilizado solo contiene canciones originales del álbum *Strangeways, Here We Come*. También aparece una canción de The Smiths en la banda sonora *The Wedding Singer*, pero se excluye porque no es un álbum de estudio.

Por esta razón, no es útil usar como visualización principal un gráfico por álbum: mostraría tres álbumes vacíos y uno con diez canciones. Para aportar más valor, los gráficos musicales se hacen a nivel de canción.

## 5. Gráfico 1 — Energy por canción

Este gráfico compara la variable `energy`, que representa la intensidad o energía musical estimada para cada canción.

In [ ]:
plot_path = PROJECT_ROOT / "outputs" / "plots" / "01_energy_por_cancion.png"
Image(filename=str(plot_path))

## 6. Gráfico 2 — Valence por canción

La variable `valence` representa la positividad musical. Valores bajos suelen indicar canciones más oscuras o melancólicas; valores altos se asocian con canciones más positivas o alegres.

In [ ]:
plot_path = PROJECT_ROOT / "outputs" / "plots" / "02_valence_por_cancion.png"
Image(filename=str(plot_path))

## 7. Gráfico 3 — Energy vs Valence

Este gráfico cruza `energy` y `valence`. Permite observar si las canciones más intensas también son más positivas, o si combinan energía alta con una sensación más melancólica.

In [ ]:
plot_path = PROJECT_ROOT / "outputs" / "plots" / "03_energy_vs_valence_por_cancion.png"
Image(filename=str(plot_path))

## 8. Análisis de letras con lyrics.ovh

Para cumplir el apartado de letras, primero se intentó recuperar letras de canciones de The Smiths usando `lyrics.ovh`.

En las pruebas realizadas, la API no devolvió letras para The Smiths. Para no dejar vacío el componente de letras, se utilizó un conjunto alternativo de canciones de varios artistas, algo permitido por el enunciado porque habla de analizar letras de **un artista o de un conjunto de artistas**.

In [ ]:
lyrics_summary_path = PROJECT_ROOT / "data" / "processed" / "lyrics_summary.csv"
lyrics_words_path = PROJECT_ROOT / "data" / "processed" / "lyrics_word_frequency.csv"

lyrics_summary = pd.read_csv(lyrics_summary_path)
lyrics_words = pd.read_csv(lyrics_words_path)

print("Letras recuperadas:")
display(lyrics_summary[["artist", "song", "source", "lyrics_length"]])

print("Palabras más frecuentes:")
display(lyrics_words)

## 9. Gráfico 4 — Palabras frecuentes en letras

Este gráfico muestra las palabras más repetidas en las letras recuperadas. Antes de contar palabras se eliminaron términos muy comunes en inglés, como `the`, `and`, `you`, etc., para que el resultado fuese más representativo.

In [ ]:
plot_path = PROJECT_ROOT / "outputs" / "plots" / "04_palabras_frecuentes_letras.png"
Image(filename=str(plot_path))

## 10. Gráfico 5 — Nube de palabras

La nube de palabras permite ver de forma visual cuáles son los términos dominantes en las letras analizadas. Las palabras que aparecen más veces se muestran con mayor tamaño.

In [ ]:
plot_path = PROJECT_ROOT / "outputs" / "plots" / "05_nube_palabras_letras.png"
Image(filename=str(plot_path))

## 11. Web app con Streamlit

Además del notebook, el proyecto incluye una web app en:

```text
app/streamlit_app.py
```

La app permite explorar:

- canciones filtradas de The Smiths;
- variables musicales por canción;
- gráfico `energy` vs `valence`;
- palabras frecuentes de letras;
- nube de palabras;
- consulta manual de letras con `lyrics.ovh`.

Para ejecutarla localmente:

```bash
streamlit run app/streamlit_app.py
```


## 12. Conclusiones

El proyecto cumple los tres apartados del enunciado:

1. **Filtrado de álbumes de estudio:** se definieron los cuatro álbumes oficiales de estudio de The Smiths y se conservaron únicamente canciones pertenecientes a esos álbumes. El dataset solo contiene canciones del álbum *Strangeways, Here We Come*, lo cual se comprobó mediante filtros exactos y verificación directa.

2. **Componente de letras:** se utilizó `lyrics.ovh` para intentar recuperar letras. Como no se encontraron letras de The Smiths, se utilizó un conjunto alternativo de artistas para construir gráficos de palabras frecuentes y nube de palabras.

3. **Web app:** se creó una app con Streamlit que integra los componentes principales del análisis.

Una decisión importante fue usar gráficos por canción en lugar de gráficos por álbum, porque el dataset solo conserva un álbum de estudio de The Smiths. Esto hace que las visualizaciones sean más útiles y fáciles de interpretar.

## 13. Exportación a HTML/PDF

Después de ejecutar todas las celdas, se puede exportar a HTML con:

```bash
jupyter nbconvert --to html notebook/Spotify_TheSmiths_Analysis_CORREGIDO.ipynb
```

También se puede abrir el HTML desde Windows con:

```bash
explorer.exe notebook/Spotify_TheSmiths_Analysis_CORREGIDO.html
```

Para PDF, se puede abrir el HTML en el navegador y usar:

```text
Imprimir → Guardar como PDF
```